In [1]:
from azure.storage.blob import ContainerClient
account_name = "dlsaggregatedprodr9"
container_name = "saas-gold-direct-data-access"
sas_token = "sp=rl&st=2025-08-06T11:15:05Z&se=2026-07-31T19:30:05Z&spr=https&sv=2024-11-04&sr=d&sig=022LIiARMF6VCOJ1xaQGH6wv04Fwqh8quWp%2BzX07S7w%3D&sdd=2"
folder_path = "data/studio_id=10720/"
#folder_path = "data/studio_id=[your studio ID]/"


blob_service_url = f"https://{account_name}.blob.core.windows.net/"

container_client = ContainerClient(
    account_url=blob_service_url,
    container_name=container_name,
    credential=sas_token
)



In [2]:
blob_client = container_client.get_blob_client('data/studio_id=10720/export_fact_sales.csv')
blob_client_wishlist = container_client.get_blob_client('data/studio_id=10720/export_fact_visibility_wishlist.csv')
#blob_client_visibility = container_client.get_blob_client('data/studio_id=10720/fact_visibility.csv')

In [3]:
import time
import pandas as pd
from io import BytesIO


t0 = time.perf_counter()
data_rev = blob_client.download_blob().readall()
data_wl = blob_client_wishlist.download_blob().readall()
#data_vis = blob_client_visibility.download_blob().readall()

t1 = time.perf_counter()

In [4]:
df = pd.read_csv(BytesIO(data_rev))
df_wl = pd.read_csv(BytesIO(data_wl))
#df_vis = pd.read_csv(BytesIO(data_vis))

t2 = time.perf_counter()

print(f"Download: {t1 - t0:.3f}s")
print(f"CSV parse: {t2 - t1:.3f}s")
print(f"Total: {t2 - t0:.3f}s")

Download: 88.887s
CSV parse: 7.824s
Total: 96.711s


/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_77553/3286014683.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_wl = pd.read_csv(BytesIO(data_wl))


In [5]:
df_wl

,date,unique_sku_id,base_sku_id,human_name,product_id,product_name,package_name,custom_group,portal_platform_region_id,portal,store,country_code,non_owner_visits,non_owner_impressions,adds,deletes,purchases_activations_gifts
0,2010-01-01,1945140-store:10720,1945140,A Memoir Blue - Original Soundtrack,A Memoir Blue - Original Soundtrack:171010:10720,A Memoir Blue - Original Soundtrack,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
1,2010-01-01,1497450-store:10720,1497450,A Memoir Blue,A Memoir Blue:171010:10720,A Memoir Blue,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
2,2010-01-01,1116060-store:10720,1116060,Ashen - Nightstorm Isle,Ashen - Nightstorm Isle DLC:171010:10720,Ashen - Nightstorm Isle DLC,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
3,2010-01-01,1202450-store:10720,1202450,Ashen - Original Soundtrack,Ashen - Original Soundtrack:171010:10720,Ashen - Original Soundtrack,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
4,2010-01-01,649950-store:10720,649950,Ashen,Ashen:171010:10720,Ashen,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1348619,2025-09-30,702680-store:10720,702680,Wattam,Wattam:171010:10720,Wattam,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,3.0,2.0,5.0
1348620,2025-09-30,1792460-store:10720,1792460,We Kill Monsters,We Kill Monsters:171010:10720,We Kill Monsters,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,7.0,3.0,0.0
1348621,2025-09-30,501300-store:10720,501300,What Remains of Edith Finch,What Remains of Edith Finch:171010:10720,What Remains of Edith Finch,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,108.0,74.0,192.0
1348622,2025-09-30,1497460-store:10720,1497460,Wheel World,Wheel World:171010:10720,Wheel World,NaN,NaN,171010,Steam,Steam,YYY,NaN,NaN,9.0,43.0,16.0


In [6]:
df_wl.head().columns

Index(['date', 'unique_sku_id', 'base_sku_id', 'human_name', 'product_id',
       'product_name', 'package_name', 'custom_group',
       'portal_platform_region_id', 'portal', 'store', 'country_code',
       'non_owner_visits', 'non_owner_impressions', 'adds', 'deletes',
       'purchases_activations_gifts'],
      dtype='object')

In [7]:
df_wl = df_wl.groupby(["date",'product_name'])[['adds','deletes','purchases_activations_gifts','non_owner_visits','non_owner_impressions']].sum().sort_values(by='date',ascending=False).reset_index()

In [8]:
df = df.groupby(["date",'product_name'])[['all_units','units_returned','units_freely_distributed','units_sold_in_retail','gross_revenue', 'gross_returned']].sum().sort_values(by='date',ascending=False).reset_index()

In [9]:
df.columns

Index(['date', 'product_name', 'all_units', 'units_returned',
       'units_freely_distributed', 'units_sold_in_retail', 'gross_revenue',
       'gross_returned'],
      dtype='object')

In [10]:
product_name = "LEGO Voyagers - Friend’s Pass"
product_name_2 = "LEGO Voyagers"

In [11]:
df.product_name.unique()

array(['What Remains of Edith Finch', 'Outer Wilds', 'Donut County',
       'Florence', 'Kentucky Route Zero: TV Edition', 'LEGO Voyagers',
       'Gorogoa', 'Outer Wilds - Echoes of the Eye DLC', 'Storyteller',
       'Stray', 'The Pathless', 'LEGO Voyagers - Soundtrack', 'I Am Dead',
       'If Found...', 'Journey', 'LEGO Voyagers - Friend’s Pass',
       'Morsels', 'Last Stop', 'Lorelei and the Laser Eyes',
       'Lorelei and the Laser Eyes - Original Soundtrack',
       'Lushfoil Photography Sim', 'Maquette', 'Hindsight', 'Mundaun',
       'Hohokum', 'Florence - Original Soundtrack',
       'Gorogoa - Original Soundtrack', 'Gone Home', 'Flower',
       'Neon White - Original Soundtrack', 'Flock',
       'Donut County - Original Soundtrack', 'Cocoon', 'Bounty Star',
       'Ashen - Original Soundtrack', 'Ashen - Nightstorm Isle DLC',
       'Ashen', 'A Memoir Blue', 'Neon White',
       'Last Stop - Original Soundtrack', 'Open Roads', 'Wanderstop',
       'The Pathless - Original S

In [12]:
df.query("product_name ==@product_name")

,date,product_name,all_units,units_returned,units_freely_distributed,units_sold_in_retail,gross_revenue,gross_returned
17,2025-09-29,LEGO Voyagers - Friend’s Pass,269,0,269,0,0.0,0.0
86,2025-09-28,LEGO Voyagers - Friend’s Pass,3096,0,3096,0,0.0,0.0
135,2025-09-27,LEGO Voyagers - Friend’s Pass,3356,0,3356,0,0.0,0.0
198,2025-09-26,LEGO Voyagers - Friend’s Pass,2203,0,2203,0,0.0,0.0
253,2025-09-25,LEGO Voyagers - Friend’s Pass,2158,0,2158,0,0.0,0.0
310,2025-09-24,LEGO Voyagers - Friend’s Pass,2096,0,2096,0,0.0,0.0
369,2025-09-23,LEGO Voyagers - Friend’s Pass,2735,0,2735,0,0.0,0.0
418,2025-09-22,LEGO Voyagers - Friend’s Pass,2715,0,2715,0,0.0,0.0
474,2025-09-21,LEGO Voyagers - Friend’s Pass,4527,0,4527,0,0.0,0.0
537,2025-09-20,LEGO Voyagers - Friend’s Pass,4738,0,4738,0,0.0,0.0


In [13]:
df.query("product_name ==@product_name")

,date,product_name,all_units,units_returned,units_freely_distributed,units_sold_in_retail,gross_revenue,gross_returned
17,2025-09-29,LEGO Voyagers - Friend’s Pass,269,0,269,0,0.0,0.0
86,2025-09-28,LEGO Voyagers - Friend’s Pass,3096,0,3096,0,0.0,0.0
135,2025-09-27,LEGO Voyagers - Friend’s Pass,3356,0,3356,0,0.0,0.0
198,2025-09-26,LEGO Voyagers - Friend’s Pass,2203,0,2203,0,0.0,0.0
253,2025-09-25,LEGO Voyagers - Friend’s Pass,2158,0,2158,0,0.0,0.0
310,2025-09-24,LEGO Voyagers - Friend’s Pass,2096,0,2096,0,0.0,0.0
369,2025-09-23,LEGO Voyagers - Friend’s Pass,2735,0,2735,0,0.0,0.0
418,2025-09-22,LEGO Voyagers - Friend’s Pass,2715,0,2715,0,0.0,0.0
474,2025-09-21,LEGO Voyagers - Friend’s Pass,4527,0,4527,0,0.0,0.0
537,2025-09-20,LEGO Voyagers - Friend’s Pass,4738,0,4738,0,0.0,0.0


In [14]:
df_wl.product_name.unique()

array(['to a T', 'Forever Ago', 'Mixtape', 'Maquette',
       'Lushfoil Photography Sim', 'Lorelei and the Laser Eyes',
       'Last Stop', 'LEGO Voyagers', 'Journey', 'If Found...',
       'I Am Dead', 'Hohokum', 'Hindsight', 'Gorogoa', 'Flower',
       'Mundaun', 'Florence', 'Flock', 'Due Process', 'Donut County',
       'Demi', 'D-topia', 'Bounty Star', 'Blade Runner 2033: Labyrinth',
       'Big Hops', 'Ashen - Nightstorm Isle DLC', 'Ashen',
       'A Memoir Blue', 'Morsels', 'Cocoon', 'Neon White', 'Telling Lies',
       'Neon White - Original Soundtrack', 'Wheel World',
       'What Remains of Edith Finch', 'We Kill Monsters', 'Wanderstop',
       'Twelve Minutes', 'Thirsty Suitors', 'The Unfinished Swan',
       'The Pathless - Original Soundtrack', 'The Pathless',
       'The Lost Wild', 'The Artful Escape', 'Wattam',
       'Stray - Original Soundtrack', 'People of Note', 'Stray',
       'Open Roads', 'Outer Wilds - Echoes of the Eye DLC',
       'Outer Wilds - Original Soundt

In [38]:
df_wl.query("product_name ==@product_name_2")['adds'].sum()

412463.0

In [48]:
data = df.merge(df_wl, on=['product_name', 'date'], how='outer')

In [50]:
data = data.sort_values(by='date',ascending=True)

In [56]:
data.query("product_name ==@product_name")['adds'].sum()

104.0

In [19]:
data.groupby("product_name")['units_returned'].sum().sort_values(ascending=False)

product_name
Stray                           258223
Outer Wilds                     254261
Journey                         126380
What Remains of Edith Finch     110516
Storyteller                      57606
                                 ...  
Blade Runner 2033: Labyrinth         0
The Lost Wild                        0
Forever Ago                          0
Bounty Star                          0
Heron                                0
Name: units_returned, Length: 101, dtype: int64

In [20]:
data.groupby("product_name")['all_units'].get_group("Wheel World").sum() - data.groupby("product_name")['units_freely_distributed'].get_group("Wheel World").sum() - data.groupby("product_name")['units_sold_in_retail'].get_group("Wheel World").sum() -data.groupby("product_name")['units_returned'].get_group("Wheel World").sum()

14897

In [21]:
data

,date,product_name,all_units,units_returned,units_freely_distributed,units_sold_in_retail,gross_revenue,gross_returned,adds,deletes,purchases_activations_gifts,non_owner_visits,non_owner_impressions
89444,2016-12-19,What Remains of Edith Finch,1,0,0,1,0.00,0.0,21.0,1.0,0.0,0.0,0.0
89443,2016-12-21,What Remains of Edith Finch,2,0,0,2,0.00,0.0,17.0,1.0,0.0,0.0,0.0
89442,2016-12-29,What Remains of Edith Finch,1,0,0,1,0.00,0.0,29.0,3.0,0.0,0.0,0.0
89441,2017-01-10,What Remains of Edith Finch,1,0,0,1,0.00,0.0,39.0,2.0,0.0,0.0,0.0
89440,2017-01-16,What Remains of Edith Finch,1,0,0,1,0.00,0.0,28.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7,2025-09-30,Outer Wilds - Echoes of the Eye DLC,2,0,0,0,16.90,0.0,49.0,10.0,34.0,0.0,0.0
9,2025-09-30,Stray,10,0,0,0,190.26,0.0,445.0,273.0,217.0,0.0,0.0
10,2025-09-30,The Pathless,1,0,0,0,8.84,0.0,14.0,16.0,22.0,0.0,0.0
5,2025-09-30,LEGO Voyagers,70,0,61,0,220.44,0.0,215.0,52.0,28.0,0.0,0.0


In [22]:
data['date'] = pd.to_datetime(data['date'])
#data['date_str'] = data['date'].dt.strftime("%Y-%m-%d")

In [23]:
data['units_excl_refunds'] = data['all_units'] - data['units_returned'] - data['units_freely_distributed']- data['units_sold_in_retail']
data['revenue_excl_refunds'] = data['gross_revenue'] - data['gross_returned']
data['units_excl_refunds_incl_free'] = data['all_units'] - data['units_returned']


In [24]:
grouped = data.groupby('product_name')

In [25]:
grouped.get_group(product_name)['units_returned'].sum()

1

In [26]:
rename_dict = {
'date':'Date', 
'product_name':'Product', 
'revenue_excl_refunds':'Revenue (excl. refunds)', 
'units_excl_refunds':'Units (excl. refunds)',
'units_excl_refunds_incl_free':'units_excl_refunds_incl_free',

'units_freely_distributed':'Free units', 
    'adds':'Wishlist adds', 
    'non_owner_visits':'Non-owner visits',
       'non_owner_impressions':'Non-owner impressions', 
'units_returned': 'Refunded units',
    'deletes':"wl_deletes",
    'purchases_activations_gifts':"wl_activations",
    
}

In [27]:
data.rename(rename_dict, axis=1, inplace=True)
#data = data.drop_duplicates(subset=['day', 'product'])
#data['day'] = pd.to_datetime(test['day'] )
#data = data.sort_values(by='day')

In [28]:
product_name = "LEGO Voyagers - Friend’s Pass"


In [29]:
data.query("Product ==@product_name")

,Date,Product,all_units,Refunded units,Free units,units_sold_in_retail,gross_revenue,gross_returned,Wishlist adds,wl_deletes,wl_activations,Non-owner visits,Non-owner impressions,Units (excl. refunds),Revenue (excl. refunds),units_excl_refunds_incl_free
1194,2025-09-08,LEGO Voyagers - Friend’s Pass,1,0,1,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0.0,1
1143,2025-09-09,LEGO Voyagers - Friend’s Pass,2,0,2,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0.0,2
814,2025-09-15,LEGO Voyagers - Friend’s Pass,3808,0,3808,0,0.0,0.0,10.0,1.0,1.0,134.0,1217.0,0,0.0,3808
755,2025-09-16,LEGO Voyagers - Friend’s Pass,5454,0,5454,0,0.0,0.0,10.0,1.0,4.0,111.0,1134.0,0,0.0,5454
698,2025-09-17,LEGO Voyagers - Friend’s Pass,4367,0,4367,0,0.0,0.0,12.0,1.0,3.0,98.0,1076.0,0,0.0,4367
646,2025-09-18,LEGO Voyagers - Friend’s Pass,3710,0,3710,0,0.0,0.0,13.0,6.0,3.0,89.0,928.0,0,0.0,3710
586,2025-09-19,LEGO Voyagers - Friend’s Pass,3696,1,3695,0,0.0,0.0,9.0,3.0,3.0,111.0,916.0,0,0.0,3695
537,2025-09-20,LEGO Voyagers - Friend’s Pass,4738,0,4738,0,0.0,0.0,11.0,2.0,1.0,82.0,912.0,0,0.0,4738
474,2025-09-21,LEGO Voyagers - Friend’s Pass,4527,0,4527,0,0.0,0.0,4.0,2.0,0.0,82.0,900.0,0,0.0,4527
418,2025-09-22,LEGO Voyagers - Friend’s Pass,2715,0,2715,0,0.0,0.0,3.0,0.0,0.0,47.0,544.0,0,0.0,2715


In [30]:
data[['Date', 'Product', 'Revenue (excl. refunds)', 'Units (excl. refunds)','units_excl_refunds_incl_free',
       'Free units', 'Wishlist adds', 'Non-owner visits',
       'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date').query("Date=='2025-09-08' & Product ==@product_name")

/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_77553/3020157460.py:3: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date').query("Date=='2025-09-08' & Product ==@product_name")


,Date,Product,Revenue (excl. refunds),Units (excl. refunds),units_excl_refunds_incl_free,Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
1194,2025-09-08,LEGO Voyagers - Friend’s Pass,0.0,0,1,1,NaN,NaN,NaN,0,NaN,NaN


In [31]:
export_df = data[['Date', 'Product', 'Revenue (excl. refunds)', 'Units (excl. refunds)','units_excl_refunds_incl_free',
       'Free units', 'Wishlist adds', 'Non-owner visits',
       'Non-owner impressions', 'Refunded units','wl_deletes','wl_activations']].sort_values(by='Date')

export_df.query("Date=='2025-09-18' & Product ==@product_name")

/var/folders/vh/qvl8gx_d2m370qv72srt5r1m0000gp/T/ipykernel_77553/3129069852.py:5: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  export_df.query("Date=='2025-09-18' & Product ==@product_name")


,Date,Product,Revenue (excl. refunds),Units (excl. refunds),units_excl_refunds_incl_free,Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
646,2025-09-18,LEGO Voyagers - Friend’s Pass,0.0,0,3710,3710,13.0,89.0,928.0,0,6.0,3.0


In [32]:
export_df.tail(25)

,Date,Product,Revenue (excl. refunds),Units (excl. refunds),units_excl_refunds_incl_free,Free units,Wishlist adds,Non-owner visits,Non-owner impressions,Refunded units,wl_deletes,wl_activations
13,2025-09-29,If Found...,193.71,46,66,0,61.0,845.0,13086.0,1,50.0,28.0
23,2025-09-29,Maquette,101.03,13,13,0,23.0,608.0,6385.0,0,35.0,7.0
22,2025-09-29,Lushfoil Photography Sim,1142.44,140,147,1,243.0,3072.0,76051.0,19,203.0,88.0
21,2025-09-29,Lorelei and the Laser Eyes - Original Soundtrack,7.31,1,1,0,0.0,76.0,3822.0,0,0.0,0.0
20,2025-09-29,Lorelei and the Laser Eyes,2575.67,162,162,0,173.0,2999.0,59222.0,0,163.0,116.0
19,2025-09-29,Last Stop,99.39,19,19,0,14.0,594.0,8898.0,0,41.0,10.0
24,2025-09-29,Hindsight,19.43,4,4,0,9.0,336.0,5583.0,0,24.0,1.0
16,2025-09-29,LEGO Voyagers,24819.36,1159,2559,1283,4214.0,32583.0,1076449.0,93,650.0,540.0
15,2025-09-29,Kentucky Route Zero: TV Edition,104.51,10,14,4,7.0,7.0,212.0,0,4.0,0.0
14,2025-09-29,Journey,3037.29,707,786,0,617.0,6607.0,242068.0,8,389.0,437.0


In [33]:
export_df.to_csv("DB_Update/API/bulkAPI_Export.csv", index=False)

In [34]:
export_df['Product'].unique()

array(['What Remains of Edith Finch',
       'What Remains of Edith Finch - Original Soundtrack', 'Gorogoa',
       'Telling Lies', 'Gorogoa - Original Soundtrack', 'Ashen',
       'Maquette', 'Florence', 'Outer Wilds', 'Journey', 'Donut County',
       'No Goblin Game 3', 'Donut County - Original Soundtrack',
       'Gone Home', 'Mundaun', 'Wattam', 'I Am Dead', 'Flower',
       'If Found...', 'Telling Lies - Original Soundtrack',
       'Ashen - Nightstorm Isle DLC', 'Sayonara Wild Hearts',
       'Ashen - Original Soundtrack',
       'Sayonara Wild Hearts - Original Soundtrack',
       'Kentucky Route Zero: TV Edition',
       'Florence - Original Soundtrack', 'Last Stop',
       'The Unfinished Swan', 'Hindsight',
       'If Found... - Original Soundtrack', 'Twelve Minutes',
       'Outer Wilds - Original Soundtrack', 'Due Process',
       'The Artful Escape', 'I Am Dead - Original Soundtrack',
       'The Pathless', 'Flock', 'Due Process - Original Soundtrack',
       'Wattam - Or

In [35]:
stop

NameError: name 'stop' is not defined

In [ ]:
stop